# Zero-shot detectors (role C): ppl + logrank
No logic here: every cell just calls `src/zeroshot.py`. Settings: GPU T4, Internet on, dataset `team20-m4-processed` attached.

In [ ]:
!git clone -b main https://github.com/NIRMAL20062/generated_text_detection.git
%cd generated_text_detection
# If pip breaks torch, comment this line out: Kaggle's preinstalled packages are enough.
!pip install -q -r requirements.txt
# The dataset mounts under /kaggle/input/... (path differs by account), so find it.
!mkdir -p data && ln -sfn "$(dirname "$(find /kaggle/input -name m4_en.parquet | head -1)")" data/processed
!ls -l data/processed/   # must show m4_en.parquet
!python -m src.data --check-splits


In [ ]:
!python -m src.zeroshot --method ppl

In [ ]:
!python -m src.zeroshot --method logrank

In [ ]:
# Fast-DetectGPT (fallback for DetectGPT): DetectGPT subset + paraphrase test ids only. Add --full to score everything.
!python -m src.zeroshot --method fastdetectgpt

In [ ]:
# Metrics for every setting with the shared evaluator.
# src/eval.py is only on the a-data branch until PR #19 merges, so fetch just that file if it is missing.
import glob, os, subprocess, sys, urllib.request
EVAL_URL = "https://raw.githubusercontent.com/NIRMAL20062/generated_text_detection/a-data/src/eval.py"
if not os.path.exists("src/eval.py"):
    urllib.request.urlretrieve(EVAL_URL, "src/eval.py")
for method in ("ppl", "logrank", "fastdetectgpt"):
    for p in sorted(glob.glob(f"results/{method}/*/preds.csv")):
        r = subprocess.run([sys.executable, "-m", "src.eval", p], capture_output=True, text=True)
        print((r.stdout.strip().splitlines() or [""])[0] if r.returncode == 0 else f"FAILED {p}: {r.stderr.strip().splitlines()[-1]}")


In [ ]:
!ls results/ppl results/logrank results/fastdetectgpt
